# CycleGAN: Photo-to-Sketch Translation

**Platform:** Kaggle (T4 x2 Dual GPU) | **Image Size:** 128×128 | **ResNet Blocks:** 6

This notebook follows the full assignment guide:
- **Part 1:** Data Preparation (Domain A: Sketch, Domain B: Photo)
- **Part 2:** Forward Pass & Training Setup
- **Part 3:** Training Loop (AMP, Checkpointing, Resume)
- **Part 4:** Visualization (5 qualitative examples)
- **Part 5:** Evaluation (SSIM, PSNR) + Loss Plots
- **Part 6:** Gradio App Deployment

## Setup: Install Dependencies & Download Dataset

In [ ]:
!pip install -q gdown gradio scikit-image
!apt-get update -q && apt-get install -q p7zip-full -y

## Part 1: Data Preparation
1. Load unpaired datasets (Domain A: Sketch, Domain B: Photo)
2. Resize all images to **128 × 128**
3. Normalize images to range **[-1, 1]**
4. Create separate DataLoaders for each domain

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import random
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import gradio as gr

# ─── Configuration ───────────────────────────────────────────────────────────
IMG_SIZE       = 128          # Reduced from 256 for T4 performance
BATCH_SIZE     = 4            # AMP-safe batch size for dual T4
LR             = 0.0002
BETAS          = (0.5, 0.999)
LAMBDA_CYCLE   = 10.0
LAMBDA_ID      = 5.0
N_BLOCKS       = 6            # Reduced from 9 to fit T4 VRAM
NUM_EPOCHS     = 10
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'latest_cyclegan.pth')
SKETCH_ROOT    = 'sketchy/sketchy/sketches'
PHOTO_ROOT     = 'sketchy/sketchy/photos'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Device: {DEVICE} | Image Size: {IMG_SIZE}x{IMG_SIZE} | Batch: {BATCH_SIZE}')

# ─── Download Dataset ────────────────────────────────────────────────────────
DRIVE_ID = '0B7ISyeE8QtDdTjE1MG9Gcy1kSkE'
if not os.path.exists('sketchy'):
    if not os.path.exists('sketchy_data.zip'):
        !gdown --id {DRIVE_ID} -O sketchy_data.zip --confirm
    !7z x sketchy_data.zip -osketchy
    print('Dataset extracted.')

# ─── 1. Unpaired Dataset ─────────────────────────────────────────────────────
class UnpairedDataset(Dataset):
    def __init__(self, root_A, root_B, transform=None):
        self.transform = transform
        self.files_A = self._get_files(root_A)
        self.files_B = self._get_files(root_B)
        print(f'Domain A (Sketch): {len(self.files_A)} | Domain B (Photo): {len(self.files_B)}')

    def _get_files(self, root):
        out = []
        for r, _, fs in os.walk(root):
            for f in fs:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    out.append(os.path.join(r, f))
        return out

    def __getitem__(self, idx):
        img_A = Image.open(self.files_A[idx % len(self.files_A)]).convert('RGB')
        img_B = Image.open(self.files_B[random.randint(0, len(self.files_B)-1)]).convert('RGB')
        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)
        return {'A': img_A, 'B': img_B}

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))

# 2. Resize + 3. Normalize to [-1, 1]
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. Separate DataLoaders
if os.path.exists(SKETCH_ROOT):
    dataset    = UnpairedDataset(SKETCH_ROOT, PHOTO_ROOT, transform=transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=4, pin_memory=True)
    print(f'DataLoader ready: {len(dataloader)} batches per epoch')
else:
    print('WARNING: Dataset not found. Run download cell first.')

## Part 2: Model Architecture & Forward Pass

**Architecture:**
- **Generator (G_AB, G_BA):** 6-block ResNet with Reflection Padding (no checkerboard artifacts)
- **Discriminator (D_A, D_B):** 70×70 PatchGAN

**Forward Pass Logic:**
1. `fake_B = G_AB(real_A)` → Translate Sketch → Photo
2. `rec_A  = G_BA(fake_B)` → Translate back Photo → Sketch (Cycle A)
3. `fake_A = G_BA(real_B)` → Translate Photo → Sketch
4. `rec_B  = G_AB(fake_A)` → Translate back Sketch → Photo (Cycle B)

**Loss Functions:**
- **D. Adversarial Loss:** MSE (LSGAN) for stability
- **A. Cycle-Consistency Loss:** L1, λ=10
- **B. Identity Loss:** L1, λ=5

**Optimizer:** Adam | LR=0.0002 | Betas=(0.5, 0.999)

In [ ]:
import torch.nn as nn
import itertools
from torch.cuda.amp import GradScaler, autocast

# ─── ResNet Generator ────────────────────────────────────────────────────────
class ResNetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, padding=0, bias=True),
            nn.InstanceNorm2d(dim), nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3, padding=0, bias=True),
            nn.InstanceNorm2d(dim)
        )
    def forward(self, x): return x + self.block(x)

class ResNetGenerator(nn.Module):
    def __init__(self, in_nc=3, out_nc=3, ngf=64, n_blocks=6):
        super().__init__()
        m = [nn.ReflectionPad2d(3),
             nn.Conv2d(in_nc, ngf, 7, bias=True), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        for i in range(2):
            f = 2**i
            m += [nn.Conv2d(ngf*f, ngf*f*2, 3, stride=2, padding=1, bias=True),
                  nn.InstanceNorm2d(ngf*f*2), nn.ReLU(True)]
        f = 4
        for _ in range(n_blocks): m += [ResNetBlock(ngf*f)]
        for i in range(2):
            f = 4 // (2**i)
            m += [nn.ConvTranspose2d(ngf*f, ngf*f//2, 3, stride=2, padding=1, output_padding=1, bias=True),
                  nn.InstanceNorm2d(ngf*f//2), nn.ReLU(True)]
        m += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, out_nc, 7), nn.Tanh()]
        self.model = nn.Sequential(*m)
    def forward(self, x): return self.model(x)

# ─── PatchGAN Discriminator ──────────────────────────────────────────────────
class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_nc=3, ndf=64, n_layers=3):
        super().__init__()
        seq = [nn.Conv2d(in_nc, ndf, 4, stride=2, padding=1), nn.LeakyReLU(0.2, True)]
        nf = 1
        for n in range(1, n_layers):
            nf_prev, nf = nf, min(2**n, 8)
            seq += [nn.Conv2d(ndf*nf_prev, ndf*nf, 4, stride=2, padding=1, bias=True),
                    nn.InstanceNorm2d(ndf*nf), nn.LeakyReLU(0.2, True)]
        nf_prev, nf = nf, min(2**n_layers, 8)
        seq += [nn.Conv2d(ndf*nf_prev, ndf*nf, 4, stride=1, padding=1, bias=True),
                nn.InstanceNorm2d(ndf*nf), nn.LeakyReLU(0.2, True)]
        seq += [nn.Conv2d(ndf*nf, 1, 4, stride=1, padding=1)]
        self.model = nn.Sequential(*seq)
    def forward(self, x): return self.model(x)

def init_weights(net, gain=0.02):
    def fn(m):
        cn = m.__class__.__name__
        if hasattr(m, 'weight') and ('Conv' in cn or 'Linear' in cn):
            nn.init.normal_(m.weight.data, 0.0, gain)
            if m.bias is not None: nn.init.constant_(m.bias.data, 0.0)
    net.apply(fn)

# ─── Trainer with Full AMP ───────────────────────────────────────────────────
class CycleGANTrainer:
    def __init__(self, device, lr=0.0002, beta1=0.5, beta2=0.999,
                 lambda_cycle=10.0, lambda_id=5.0):
        self.device       = device
        self.lambda_cycle = lambda_cycle
        self.lambda_id    = lambda_id

        self.G_AB = ResNetGenerator(n_blocks=N_BLOCKS).to(device)
        self.G_BA = ResNetGenerator(n_blocks=N_BLOCKS).to(device)
        self.D_A  = PatchGANDiscriminator().to(device)
        self.D_B  = PatchGANDiscriminator().to(device)
        for net in [self.G_AB, self.G_BA, self.D_A, self.D_B]: init_weights(net)

        self.crit_GAN  = nn.MSELoss()
        self.crit_cycle = nn.L1Loss()
        self.crit_id   = nn.L1Loss()

        self.optimizer_G   = torch.optim.Adam(
            itertools.chain(self.G_AB.parameters(), self.G_BA.parameters()),
            lr=lr, betas=(beta1, beta2))
        self.optimizer_D_A = torch.optim.Adam(self.D_A.parameters(), lr=lr, betas=(beta1, beta2))
        self.optimizer_D_B = torch.optim.Adam(self.D_B.parameters(), lr=lr, betas=(beta1, beta2))

        self.scaler_G  = GradScaler()
        self.scaler_DA = GradScaler()
        self.scaler_DB = GradScaler()

    def train_step(self, real_A, real_B):
        ones_B = torch.ones_like(self.D_B(self.G_AB(real_A)).detach()).to(self.device)
        ones_A = torch.ones_like(self.D_A(self.G_BA(real_B)).detach()).to(self.device)

        # ── Generators ──────────────────────────────────────────────────────
        self.optimizer_G.zero_grad()
        with autocast():
            fake_B = self.G_AB(real_A)                          # Sketch → Photo
            rec_A  = self.G_BA(fake_B)                          # Photo  → Sketch (cycle)
            fake_A = self.G_BA(real_B)                          # Photo  → Sketch
            rec_B  = self.G_AB(fake_A)                          # Sketch → Photo  (cycle)

            # D. Adversarial loss
            loss_GAN = (self.crit_GAN(self.D_B(fake_B), torch.ones_like(self.D_B(fake_B)))
                      + self.crit_GAN(self.D_A(fake_A), torch.ones_like(self.D_A(fake_A))))
            # A. Cycle-consistency loss
            loss_cycle = (self.crit_cycle(rec_A, real_A)
                        + self.crit_cycle(rec_B, real_B)) * self.lambda_cycle
            # B. Identity loss
            loss_id = (self.crit_id(self.G_BA(real_A), real_A)
                     + self.crit_id(self.G_AB(real_B), real_B)) * self.lambda_id
            loss_G = loss_GAN + loss_cycle + loss_id

        self.scaler_G.scale(loss_G).backward()
        self.scaler_G.step(self.optimizer_G)
        self.scaler_G.update()

        # ── Discriminator A ──────────────────────────────────────────────────
        self.optimizer_D_A.zero_grad()
        with autocast():
            loss_D_A = (self.crit_GAN(self.D_A(real_A), torch.ones_like(self.D_A(real_A)))
                      + self.crit_GAN(self.D_A(fake_A.detach()), torch.zeros_like(self.D_A(fake_A.detach())))) * 0.5
        self.scaler_DA.scale(loss_D_A).backward()
        self.scaler_DA.step(self.optimizer_D_A)
        self.scaler_DA.update()

        # ── Discriminator B ──────────────────────────────────────────────────
        self.optimizer_D_B.zero_grad()
        with autocast():
            loss_D_B = (self.crit_GAN(self.D_B(real_B), torch.ones_like(self.D_B(real_B)))
                      + self.crit_GAN(self.D_B(fake_B.detach()), torch.zeros_like(self.D_B(fake_B.detach())))) * 0.5
        self.scaler_DB.scale(loss_D_B).backward()
        self.scaler_DB.step(self.optimizer_D_B)
        self.scaler_DB.update()

        return {
            'loss_G':     loss_G.item(),
            'loss_D':     (loss_D_A + loss_D_B).item(),
            'loss_cycle': loss_cycle.item()
        }

trainer = CycleGANTrainer(DEVICE, lr=LR, beta1=BETAS[0], beta2=BETAS[1],
                          lambda_cycle=LAMBDA_CYCLE, lambda_id=LAMBDA_ID)
print('Trainer initialized. Generators have', N_BLOCKS, 'ResNet blocks.')

## Part 3: Training Setup & Main Loop
**Strategy:** Train both generators and discriminators alternately | Monitor cycle loss closely.

**Required Techniques (Kaggle T4×2):**
- Mixed Precision (`torch.cuda.amp`)
- Batch Size: 4–8
- Reduce image size (128×128 ✓)
- Reduce ResNet blocks (6 ✓)
- Save checkpoints frequently ✓

In [ ]:
history     = {'G_loss': [], 'D_loss': [], 'Cyc_loss': []}
start_epoch = 0

# ─── Resume Capability ───────────────────────────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    print(f'Loading checkpoint from {CHECKPOINT_PATH} ...')
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    trainer.G_AB.load_state_dict(ckpt['G_AB'])
    trainer.G_BA.load_state_dict(ckpt['G_BA'])
    trainer.D_A.load_state_dict(ckpt['D_A'])
    trainer.D_B.load_state_dict(ckpt['D_B'])
    trainer.optimizer_G.load_state_dict(ckpt['optimizer_G'])
    trainer.optimizer_D_A.load_state_dict(ckpt['optimizer_D_A'])
    trainer.optimizer_D_B.load_state_dict(ckpt['optimizer_D_B'])
    start_epoch = ckpt['epoch'] + 1
    history     = ckpt.get('history', history)
    print(f'Resumed from epoch {start_epoch}')
else:
    print('No checkpoint found. Starting fresh session.')

# ─── Training Loop ───────────────────────────────────────────────────────────
print('Starting Training Loop...')
for epoch in range(start_epoch, NUM_EPOCHS):
    g_sum = d_sum = c_sum = 0.0

    for i, batch in enumerate(dataloader):
        real_A = batch['A'].to(DEVICE)
        real_B = batch['B'].to(DEVICE)
        losses = trainer.train_step(real_A, real_B)

        g_sum += losses['loss_G']
        d_sum += losses['loss_D']
        c_sum += losses['loss_cycle']

        if i % 100 == 0:
            print(f'[E {epoch}/{NUM_EPOCHS}][B {i}/{len(dataloader)}]'
                  f" G: {losses['loss_G']:.4f}"
                  f" D: {losses['loss_D']:.4f}"
                  f" Cyc: {losses['loss_cycle']:.4f}")

    n = len(dataloader)
    history['G_loss'].append(g_sum / n)
    history['D_loss'].append(d_sum / n)
    history['Cyc_loss'].append(c_sum / n)

    # ── Save Checkpoint ──────────────────────────────────────────────────────
    torch.save({
        'epoch':         epoch,
        'G_AB':          trainer.G_AB.state_dict(),
        'G_BA':          trainer.G_BA.state_dict(),
        'D_A':           trainer.D_A.state_dict(),
        'D_B':           trainer.D_B.state_dict(),
        'optimizer_G':   trainer.optimizer_G.state_dict(),
        'optimizer_D_A': trainer.optimizer_D_A.state_dict(),
        'optimizer_D_B': trainer.optimizer_D_B.state_dict(),
        'history':       history
    }, CHECKPOINT_PATH)
    print(f'Epoch {epoch} done | Avg G: {g_sum/n:.4f} D: {d_sum/n:.4f} Cyc: {c_sum/n:.4f} | Checkpoint saved.')

## Part 4: Visualization Module
Displays **5 qualitative examples**: Input Sketch → Generated Photo → Reconstructed Sketch

In [ ]:
def denorm(t):
    """Convert tensor [-1,1] to displayable numpy [0,1] image."""
    return (t.cpu().detach().numpy().transpose(1, 2, 0) * 0.5 + 0.5).clip(0, 1)

def visualize_results(trainer, dataloader, n_samples=5):
    trainer.G_AB.eval()
    trainer.G_BA.eval()
    batch  = next(iter(dataloader))
    real_A = batch['A'][:n_samples].to(DEVICE)
    real_B = batch['B'][:n_samples].to(DEVICE)

    with torch.no_grad():
        fake_B = trainer.G_AB(real_A)   # Sketch -> Photo
        rec_A  = trainer.G_BA(fake_B)   # Reconstructed Sketch

    fig, axes = plt.subplots(n_samples, 3, figsize=(15, n_samples * 3.2))
    cols = ['Input Sketch (Domain A)', 'Generated Photo (G_AB)', 'Reconstructed Sketch (G_BA)']
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=12, fontweight='bold')

    for i in range(n_samples):
        axes[i][0].imshow(denorm(real_A[i]))
        axes[i][1].imshow(denorm(fake_B[i]))
        axes[i][2].imshow(denorm(rec_A[i]))
        for ax in axes[i]: ax.axis('off')

    plt.suptitle('CycleGAN Qualitative Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/qualitative_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Visualization saved to /kaggle/working/qualitative_results.png')
    trainer.G_AB.train()
    trainer.G_BA.train()

visualize_results(trainer, dataloader)

## Part 5: Training Logs & Quantitative Evaluation
1. **Plot** Generator Loss, Discriminator Loss, and Cycle-Consistency Loss vs Epochs.
2. **Quantitative Metrics:** SSIM and PSNR on reconstructed images.

In [ ]:
# ─── 1. Loss Plots ───────────────────────────────────────────────────────────
if history['G_loss']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    pairs = [('G_loss', 'Generator Loss', 'royalblue'),
             ('D_loss', 'Discriminator Loss', 'tomato'),
             ('Cyc_loss', 'Cycle-Consistency Loss', 'seagreen')]
    for ax, (key, title, color) in zip(axes, pairs):
        ax.plot(history[key], color=color, linewidth=2)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.grid(True, alpha=0.3)
    plt.suptitle('Training Losses vs Epochs', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_plots.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history available yet. Run the training cell first.')

# ─── 2. Quantitative Evaluation (SSIM + PSNR) ────────────────────────────────
def evaluate_model(trainer, dataloader, n_batches=3):
    trainer.G_AB.eval()
    trainer.G_BA.eval()
    all_ssim, all_psnr = [], []

    for idx, batch in enumerate(dataloader):
        if idx >= n_batches:
            break
        real_A = batch['A'].to(DEVICE)
        with torch.no_grad():
            rec_A = trainer.G_BA(trainer.G_AB(real_A))   # Full cycle

        for r, f in zip(real_A, rec_A):
            r_np = denorm(r)
            f_np = denorm(f)
            # channel_axis=-1 works for skimage >= 0.19; multichannel works for older
            try:
                s = ssim_metric(r_np, f_np, channel_axis=-1, data_range=1.0)
            except TypeError:
                s = ssim_metric(r_np, f_np, multichannel=True, data_range=1.0)
            p = psnr_metric(r_np, f_np, data_range=1.0)
            all_ssim.append(s)
            all_psnr.append(p)

    trainer.G_AB.train()
    trainer.G_BA.train()
    print('─' * 40)
    print('  Quantitative Evaluation Results')
    print('─' * 40)
    print(f'  Average SSIM : {np.mean(all_ssim):.4f}  (higher = better, max 1.0)')
    print(f'  Average PSNR : {np.mean(all_psnr):.2f} dB  (higher = better)')
    print('─' * 40)
    return np.mean(all_ssim), np.mean(all_psnr)

evaluate_model(trainer, dataloader)

## Part 6: App Deployment (Gradio)
Upload a sketch or photo — the app translates it in real-time using the trained generators.

In [ ]:
def predict(img, direction):
    """Accepts image input, performs domain translation, displays output."""
    img_t = transform(Image.fromarray(img).convert('RGB')).unsqueeze(0).to(DEVICE)
    trainer.G_AB.eval()
    trainer.G_BA.eval()
    with torch.no_grad():
        out = trainer.G_AB(img_t) if direction == 'Sketch → Photo' else trainer.G_BA(img_t)
    return (denorm(out.squeeze(0)) * 255).astype(np.uint8)

gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type='numpy', label='Input Image'),
        gr.Radio(['Sketch → Photo', 'Photo → Sketch'],
                 value='Sketch → Photo', label='Translation Direction')
    ],
    outputs=gr.Image(type='numpy', label='Generated Output'),
    title='CycleGAN: Sketch ↔ Photo Translator',
    description='Upload an image and select a translation direction. '
                'The model performs real-time domain translation using a trained CycleGAN.'
).launch(share=True)